# 19 — Audio Evaluation Metrics

## Objective

In this notebook, we will explore several metrics used to evaluate speech and audio AI systems.

We will focus on:

- Word Error Rate (WER);
- Character Error Rate (CER);
- Mean Opinion Score (MOS);
- speaker similarity using cosine similarity;
- latency-oriented evaluation;
- comparing different audio systems with multiple metrics.

## Why evaluation matters

Different audio tasks require different evaluation metrics.

For example:

ASR
→ WER / CER

TTS
→ MOS

Voice Cloning
→ Speaker Similarity

Speaker Verification
→ EER

Diarization
→ DER

Music Generation
→ FAD / CLAP

Streaming Speech-to-Speech
→ Latency

A single metric is usually not enough to evaluate an audio system correctly.

## Word Error Rate (WER)

Word Error Rate measures how different a predicted transcription is from the reference transcription.

It is based on three types of errors:

- Substitutions
- Deletions
- Insertions

Formula:

WER = (S + D + I) / N

where:

- S = substitutions
- D = deletions
- I = insertions
- N = number of words in the reference transcription

Lower WER is better.

In [1]:
reference = "the weather is sunny today"
prediction = "the weather was sunny"

reference_words = reference.split()
prediction_words = prediction.split()

print("Reference:", reference_words)
print("Prediction:", prediction_words)

Reference: ['the', 'weather', 'is', 'sunny', 'today']
Prediction: ['the', 'weather', 'was', 'sunny']


In [2]:
def word_error_rate(reference, prediction):
    ref = reference.split()
    hyp = prediction.split()

    dp = [[0] * (len(hyp) + 1) for _ in range(len(ref) + 1)]

    for i in range(len(ref) + 1):
        dp[i][0] = i

    for j in range(len(hyp) + 1):
        dp[0][j] = j

    for i in range(1, len(ref) + 1):
        for j in range(1, len(hyp) + 1):

            if ref[i - 1] == hyp[j - 1]:
                cost = 0
            else:
                cost = 1

            dp[i][j] = min(
                dp[i - 1][j] + 1,      # deletion
                dp[i][j - 1] + 1,      # insertion
                dp[i - 1][j - 1] + cost  # substitution
            )

    errors = dp[-1][-1]

    return errors / len(ref)


wer = word_error_rate(reference, prediction)

print("WER:", wer)
print("WER (%):", round(wer * 100, 2))

WER: 0.4
WER (%): 40.0


### WER example

Reference:

`the weather is sunny today`

Prediction:

`the weather was sunny`

Errors:

- `is` → `was`: substitution
- `today`: deletion

Therefore:

WER = (1 substitution + 1 deletion) / 5 reference words

WER = 0.40 = 40%

A lower WER indicates a more accurate speech recognition system.

## Character Error Rate (CER)

Character Error Rate is similar to WER, but it measures errors at the character level instead of the word level.

Formula:

CER = (S + D + I) / N

where:

- S = character substitutions
- D = character deletions
- I = character insertions
- N = number of characters in the reference

Lower CER is better.

In [3]:
def character_error_rate(reference, prediction):
    ref = list(reference)
    hyp = list(prediction)

    dp = [[0] * (len(hyp) + 1) for _ in range(len(ref) + 1)]

    for i in range(len(ref) + 1):
        dp[i][0] = i

    for j in range(len(hyp) + 1):
        dp[0][j] = j

    for i in range(1, len(ref) + 1):
        for j in range(1, len(hyp) + 1):

            if ref[i - 1] == hyp[j - 1]:
                cost = 0
            else:
                cost = 1

            dp[i][j] = min(
                dp[i - 1][j] + 1,        # deletion
                dp[i][j - 1] + 1,        # insertion
                dp[i - 1][j - 1] + cost  # substitution
            )

    errors = dp[-1][-1]

    return errors / len(ref)


cer = character_error_rate(reference, prediction)

print("CER:", cer)
print("CER (%):", round(cer * 100, 2))

CER: 0.3076923076923077
CER (%): 30.77


### CER example

Using the same transcription pair:

Reference:

`the weather is sunny today`

Prediction:

`the weather was sunny`

The Character Error Rate is:

CER = 0.3077 = 30.77%

CER evaluates transcription quality at the character level.

This can be useful for:

- short words;
- spelling-sensitive tasks;
- languages where word segmentation is less straightforward.

Lower CER indicates better transcription accuracy.

## Mean Opinion Score (MOS)

Mean Opinion Score is a subjective metric commonly used to evaluate the perceived quality and naturalness of synthesized speech.

Listeners usually rate audio on a scale from 1 to 5:

- 1 = Bad
- 2 = Poor
- 3 = Fair
- 4 = Good
- 5 = Excellent

The MOS is simply the average of all listener ratings.

Higher MOS is better.

In [4]:
mos_scores = [4, 5, 4, 3, 4, 5, 4, 4, 3, 5]

mos = sum(mos_scores) / len(mos_scores)

print("MOS scores:", mos_scores)
print("Mean Opinion Score:", round(mos, 2))

MOS scores: [4, 5, 4, 3, 4, 5, 4, 4, 3, 5]
Mean Opinion Score: 4.1


### MOS example

In this example, ten listeners rated the generated audio.

The average score is:

MOS = 4.1 / 5

This indicates that the audio was perceived as generally good to very good.

Unlike WER and CER, MOS is subjective because it depends on human perception.

## Speaker Similarity with Cosine Similarity

Voice cloning systems are often evaluated by comparing the speaker embedding of a reference voice with the speaker embedding of a generated voice.

A common metric is cosine similarity.

Cosine similarity measures the angle between two embedding vectors.

Interpretation:

- close to 1 → very similar
- close to 0 → weak similarity
- close to -1 → opposite directions

Higher similarity generally means the generated voice is closer to the target speaker.

In [5]:
import numpy as np

reference_embedding = np.array([0.8, 0.4, 0.3, 0.5])
generated_embedding = np.array([0.75, 0.42, 0.28, 0.52])

def cosine_similarity(a, b):
    return np.dot(a, b) / (
        np.linalg.norm(a) * np.linalg.norm(b)
    )

similarity = cosine_similarity(
    reference_embedding,
    generated_embedding
)

print("Speaker cosine similarity:", round(similarity, 4))

Speaker cosine similarity: 0.9986


### Speaker similarity result

The cosine similarity between the reference speaker embedding and the generated speaker embedding is:

Speaker similarity = 0.9986

Because the score is very close to 1, the two embeddings are highly similar.

In a voice cloning system, a high cosine similarity suggests that the generated voice preserves characteristics of the target speaker.

However, speaker similarity alone does not measure speech quality or intelligibility.

## Latency Evaluation

Latency is a critical metric for real-time speech and audio systems.

Examples include:

- voice assistants;
- streaming ASR;
- speech-to-speech systems;
- live translation;
- conversational AI.

Common latency measurements include:

- average latency;
- P50 latency;
- P95 latency;
- P99 latency.

Lower latency generally improves responsiveness and user experience.

In [6]:
latencies_ms = np.array([
    120, 135, 110, 140, 150,
    125, 130, 118, 145, 160
])

average_latency = np.mean(latencies_ms)
p50_latency = np.percentile(latencies_ms, 50)
p95_latency = np.percentile(latencies_ms, 95)
p99_latency = np.percentile(latencies_ms, 99)

print("Average latency:", round(average_latency, 2), "ms")
print("P50 latency:", round(p50_latency, 2), "ms")
print("P95 latency:", round(p95_latency, 2), "ms")
print("P99 latency:", round(p99_latency, 2), "ms")

Average latency: 133.3 ms
P50 latency: 132.5 ms
P95 latency: 155.5 ms
P99 latency: 159.1 ms


### Latency results

The simulated system produced the following latency measurements:

- Average latency: 133.3 ms
- P50 latency: 132.5 ms
- P95 latency: 155.5 ms
- P99 latency: 159.1 ms

The median latency is close to the average, which suggests relatively stable performance.

P95 and P99 are useful because they reveal slower responses that may not be visible when looking only at the average.

For real-time speech systems, both typical latency and worst-case latency are important.

## Multi-Metric System Comparison

Audio AI systems should rarely be evaluated using only one metric.

We will compare three hypothetical systems using:

- WER for transcription accuracy;
- MOS for perceived audio quality;
- speaker similarity;
- latency.

The best system depends on the application and the trade-offs between quality, accuracy, identity preservation, and speed.

In [7]:
import pandas as pd

results = pd.DataFrame({
    "System": ["System A", "System B", "System C"],
    "WER (%)": [8.5, 12.0, 6.8],
    "MOS": [4.1, 4.5, 3.9],
    "Speaker Similarity": [0.91, 0.96, 0.88],
    "Latency (ms)": [140, 220, 105]
})

results

,System,WER (%),MOS,Speaker Similarity,Latency (ms)
0,System A,8.5,4.1,0.91,140
1,System B,12.0,4.5,0.96,220
2,System C,6.8,3.9,0.88,105
